# Asset Rental Agreement Analytics

This notebook analyzes the Jan LAR active lease asset report and builds a prediction model for `Asset Rental Amount`.

## Section I: Accessing the Data

We load the Excel file, inspect the dataset, and define the target variable.

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

DATA_PATH = Path("../data/Jan_LAR.xlsx")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
df = pd.read_csv('/Users/prakashghatani/Desktop/Regis/MSDS692_Practicum 1/WEEK1/data/Jan LAR.csv')
print(df.shape)
df.head()

(1727, 33)


,Service Tag,Account Name,Bill to Address 1,Bill to Address 2,Bill to Address 3,Bill to City,Bill to State,Bill to Postal Code,Contract Number,Commencement Date,...,Service Tag.1,PO Number,Cost Center,Asset Address 1,Asset Address 2,Asset City,Asset State,Asset Postal Code,Asset Rental Amount,Asset Rental Tax
0,DXKB2F3,LEXMARK INTERNATIONAL INC.,PO Box 14077,NaN,NaN,LEXINGTON,KY,40512.0,001-6428292-275,09/01/2021,...,DXKB2F3,2.000219e+09,NaN,740 WEST NEW CIR RD,LEX-5050,LEXINGTON,Kentucky,40511-1806,22.81,0.0
1,7SVG2F3,LEXMARK INTERNATIONAL INC.,PO Box 14077,NaN,NaN,LEXINGTON,KY,40512.0,001-6428292-275,09/01/2021,...,7SVG2F3,2.000219e+09,NaN,740 WEST NEW CIR RD,LEX-5050,LEXINGTON,Kentucky,40511-1806,22.80,0.0
2,CNXQ2F3,LEXMARK INTERNATIONAL INC.,PO Box 14077,NaN,NaN,LEXINGTON,KY,40512.0,001-6428292-275,09/01/2021,...,CNXQ2F3,2.000219e+09,NaN,740 WEST NEW CIR RD,LEX-5050,LEXINGTON,Kentucky,40511-1806,22.81,0.0
3,FNTH2F3,LEXMARK INTERNATIONAL INC.,PO Box 14077,NaN,NaN,LEXINGTON,KY,40512.0,001-6428292-275,09/01/2021,...,FNTH2F3,2.000219e+09,NaN,740 WEST NEW CIR RD,LEX-5050,LEXINGTON,Kentucky,40511-1806,22.80,0.0
4,CDHH2F3,LEXMARK INTERNATIONAL INC.,PO Box 14077,NaN,NaN,LEXINGTON,KY,40512.0,001-6428292-275,09/01/2021,...,CDHH2F3,2.000219e+09,NaN,740 WEST NEW CIR RD,LEX-5050,LEXINGTON,Kentucky,40511-1806,22.81,0.0


In [4]:
# Basic dataset information
summary = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str),
    "Missing_Count": df.isna().sum().values,
    "Missing_Percent": (df.isna().mean().values * 100).round(2)
})
summary.sort_values("Missing_Percent", ascending=False).head(15)

,Column,Data_Type,Missing_Count,Missing_Percent
Bill to Address 2,Bill to Address 2,float64,1727,100.00
Bill to Address 3,Bill to Address 3,float64,1727,100.00
Cost Center,Cost Center,float64,1727,100.00
Rental Payment,Rental Payment,float64,1698,98.32
Asset Address 2,Asset Address 2,object,11,0.64
Service Tag,Service Tag,object,10,0.58
Service Tag.1,Service Tag.1,object,10,0.58
PO Number,PO Number,float64,3,0.17
Model Number,Model Number,object,3,0.17
Currency,Currency,object,3,0.17


In [5]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Unique Contracts:", df["Contract Number"].nunique())
print("Unique Vendors:", df["Vendor Name"].nunique())
print("Unique Manufacturers:", df["Manufacturer"].nunique())
print("Total Asset Cost:", round(df["Asset Cost"].sum(), 2))
print("Total Monthly Rental Amount:", round(df["Asset Rental Amount"].sum(), 2))

Rows: 1727
Columns: 33
Unique Contracts: 29
Unique Vendors: 1
Unique Manufacturers: 1
Total Asset Cost: 2055670.82
Total Monthly Rental Amount: 171807.61


## Section II: Exploratory Data Analysis

In [6]:
# Descriptive statistics for important numeric columns
num_cols = ["Term", "Rental Payment", "Asset Quantity", "Asset Cost", "Asset Rental Amount", "Asset Rental Tax"]
df[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Term,1724.0,36.000000,0.000000,36.00,36.00,36.00,36.00,36.00
Rental Payment,29.0,5924.400345,12195.477628,26.73,159.65,432.05,1595.70,44433.15
Asset Quantity,1724.0,1.015661,0.253332,1.00,1.00,1.00,1.00,6.00
Asset Cost,1724.0,1192.384466,221.087690,75.98,1074.52,1104.47,1281.95,1884.00
Asset Rental Amount,1724.0,99.656386,24.797347,2.82,94.26,96.88,112.43,166.02
Asset Rental Tax,1724.0,0.037117,0.677585,0.00,0.00,0.00,0.00,13.44


In [7]:
missing_df = summary.sort_values("Missing_Count", ascending=False).head(15)
fig = px.bar(missing_df, x="Column", y="Missing_Count", title="Top Missing Value Columns")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [8]:
vendor_cost = (df.groupby("Vendor Name", dropna=False)["Asset Rental Amount"]
                 .sum()
                 .reset_index()
                 .sort_values("Asset Rental Amount", ascending=False)
                 .head(10))
fig = px.bar(vendor_cost, x="Vendor Name", y="Asset Rental Amount", title="Top Vendors by Monthly Rental Amount")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [9]:
manufacturer_cost = (df.groupby("Manufacturer", dropna=False)["Asset Rental Amount"]
                     .sum()
                     .reset_index()
                     .sort_values("Asset Rental Amount", ascending=False)
                     .head(10))
fig = px.bar(manufacturer_cost, x="Manufacturer", y="Asset Rental Amount", title="Top Manufacturers by Monthly Rental Amount")
fig.show()

In [10]:
fig = px.histogram(df, x="Asset Rental Amount", nbins=50, title="Distribution of Asset Rental Amount")
fig.show()

In [11]:
fig = px.scatter(
    df,
    x="Asset Cost",
    y="Asset Rental Amount",
    color="Manufacturer",
    hover_data=["Vendor Name", "Asset Description", "Contract Number"],
    title="Asset Cost vs Monthly Rental Amount"
)
fig.show()

## Section III: Prepare Data for Training

In [12]:
# Clean column names
model_df = df.copy()
model_df.columns = (model_df.columns
                    .str.strip()
                    .str.lower()
                    .str.replace(" ", "_", regex=False)
                    .str.replace(".", "", regex=False))

# Date features
model_df["commencement_date"] = pd.to_datetime(model_df["commencement_date"], errors="coerce")
model_df["primary_term_date"] = pd.to_datetime(model_df["primary_term_date"], errors="coerce")
model_df["commencement_year"] = model_df["commencement_date"].dt.year
model_df["lease_age_days"] = (pd.Timestamp.today().normalize() - model_df["commencement_date"]).dt.days
model_df["days_until_primary_term_end"] = (model_df["primary_term_date"] - pd.Timestamp.today().normalize()).dt.days
model_df["rental_to_cost_ratio"] = model_df["asset_rental_amount"] / model_df["asset_cost"]

# Remove rows with missing target
model_df = model_df.dropna(subset=["asset_rental_amount"])
model_df[["asset_rental_amount", "asset_cost", "rental_to_cost_ratio", "commencement_year", "lease_age_days"]].head()

,asset_rental_amount,asset_cost,rental_to_cost_ratio,commencement_year,lease_age_days
0,22.81,862.35,0.026451,2021.0,1738.0
1,22.80,862.35,0.026439,2021.0,1738.0
2,22.81,862.35,0.026451,2021.0,1738.0
3,22.80,862.35,0.026439,2021.0,1738.0
4,22.81,862.60,0.026443,2021.0,1738.0


In [13]:
target = "asset_rental_amount"
features = [
    "term", "asset_quantity", "asset_cost", "asset_rental_tax",
    "commencement_year", "lease_age_days", "days_until_primary_term_end",
    "vendor_name", "manufacturer", "asset_description", "model_number",
    "contract_type", "asset_state", "asset_city"
]

features = [c for c in features if c in model_df.columns]
X = model_df[features]
y = model_df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['term', 'asset_quantity', 'asset_cost', 'asset_rental_tax', 'commencement_year', 'lease_age_days', 'days_until_primary_term_end']
Categorical features: ['vendor_name', 'manufacturer', 'asset_description', 'model_number', 'contract_type', 'asset_state', 'asset_city']


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## Section IV: Regression Modeling

In [15]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, max_depth=12)
}

results = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": rmse, # Root Mean Squared Error
        "R2": r2_score(y_test, pred)
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df

,Model,MAE,RMSE,R2
1,Random Forest,0.201415,1.823696,0.994952
0,Linear Regression,5.514447,9.190512,0.871802


## Section V: Model Evaluation and Visualization

In [16]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
y_pred = best_model.predict(X_test)

pred_df = pd.DataFrame({
    "Actual_Rental_Amount": y_test,
    "Predicted_Rental_Amount": y_pred,
    "Residual": y_test - y_pred
})

fig = px.scatter(
    pred_df,
    x="Actual_Rental_Amount",
    y="Predicted_Rental_Amount",
    title=f"Actual vs Predicted Rental Amount: {best_model_name}",
    trendline="ols"
)
fig.add_trace(
    go.Scatter(
        x=[pred_df["Actual_Rental_Amount"].min(), pred_df["Actual_Rental_Amount"].max()],
        y=[pred_df["Actual_Rental_Amount"].min(), pred_df["Actual_Rental_Amount"].max()],
        mode="lines",
        name="Perfect Prediction"
    )
)
fig.show()

In [17]:
fig = px.histogram(pred_df, x="Residual", nbins=50, title="Residual Error Distribution")
fig.show()

In [18]:
# Feature importance for Random Forest
if "Random Forest" in fitted_models:
    rf_pipe = fitted_models["Random Forest"]
    pre = rf_pipe.named_steps["preprocessor"]
    model = rf_pipe.named_steps["model"]
    feature_names = pre.get_feature_names_out()
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": model.feature_importances_
    }).sort_values("Importance", ascending=False).head(20)

    fig = px.bar(importance_df, x="Importance", y="Feature", orientation="h", title="Top 20 Feature Importances")
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.show()
    display(importance_df)

,Feature,Importance
5,num__lease_age_days,0.507834
2,num__asset_cost,0.430151
6,num__days_until_primary_term_end,0.012454
15,cat__asset_description_Dell Mobile Precision W...,0.012018
4,num__commencement_year,0.010594
34,cat__model_number_210-BLNG,0.009424
12,cat__asset_description_Dell Latitude 5430 XCTO...,0.002382
30,cat__model_number_210-BDGV,0.002073
31,cat__model_number_210-BDTV,0.001671
14,cat__asset_description_Dell Mobile Precision W...,0.001602


In [19]:
# Save model and predictions
joblib.dump(best_model, OUTPUT_DIR / "best_asset_rental_model.joblib")
pred_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
results_df.to_csv(OUTPUT_DIR / "model_results.csv", index=False)
print("Saved model and output files to:", OUTPUT_DIR)

Saved model and output files to: ../outputs


## Section VI: Business Recommendations

1. **Monitor high-cost vendors and manufacturers** because they create the largest rental exposure.
2. **Review assets with high rental-to-cost ratio** because these may indicate pricing anomalies or unfavorable lease terms.
3. **Improve missing data quality**, especially fields with very high missing percentages.
4. **Use the prediction model as a screening tool** to compare expected rental amount with actual rental amount.
5. **Build a renewal dashboard** using primary term dates to track contracts before expiration.